<a href="https://colab.research.google.com/github/leo-aguiar/ibm_hr_analytics_employee/blob/main/IBM_Analytics_Employee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Bibliotecas para manipulação, visualização e ingestão de dados
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub as kagg
import os

In [2]:
# Realiza a ingestão do dataset diretamente do Kaggle para o ambiente local
path = kagg.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")

Using Colab cache for faster access to the 'ibm-hr-analytics-attrition-dataset' dataset.


In [3]:
# Carrega o dataset de rotatividade de funcionários em um DataFrame para posterior análise exploratória e modelagem
df = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-HR-Employee-Attrition.csv"))

In [4]:
# Seleciona amostra aleatória de registros para inspeção inicial dos dados
df.sample(5)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
659,28,No,Travel_Rarely,821,Sales,5,4,Medical,1,916,...,2,80,0,4,3,3,4,2,0,2
1105,33,No,Travel_Rarely,1242,Sales,8,4,Life Sciences,1,1560,...,4,80,1,8,6,1,2,2,2,2
1406,54,No,Travel_Rarely,157,Research & Development,10,3,Medical,1,1980,...,4,80,0,9,3,3,5,2,1,4
287,38,No,Travel_Rarely,688,Research & Development,23,4,Life Sciences,1,393,...,2,80,1,10,2,3,2,2,1,2
113,24,No,Travel_Rarely,1127,Research & Development,18,1,Life Sciences,1,150,...,3,80,1,6,2,3,5,3,1,2


In [5]:
# Inspeciona a estrutura geral do dataset para avaliar volume de registros, tipos de dados, completude das colunas e possíveis necessidades de pré-processamento
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [7]:
# Avalia a completude do dataset ao consolidar, por coluna,
# a quantidade absoluta e percentual de valores ausentes por vaariável
missing_table = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().mean() * 100).round(2)
})
missing_table = missing_table.sort_values(by='Missing %', ascending=False)
missing_table

,Missing Count,Missing %
Age,0,0.0
Attrition,0,0.0
BusinessTravel,0,0.0
DailyRate,0,0.0
Department,0,0.0
DistanceFromHome,0,0.0
Education,0,0.0
EducationField,0,0.0
EmployeeCount,0,0.0
EmployeeNumber,0,0.0


In [9]:
# Annotation: Não foram identificados valores ausentes no dataset, indicando completude total das variáveis e eliminando a necessidade de tartamento de missinbg values nesta etapa do pré-processamento.

In [14]:
# Segmenta as variáveis conforme seu tipo de dado para direcionar análises específicas de atributos numéricos e categóricos
numerical_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

In [16]:
numerical_cols

['Age',
 'DailyRate',
 'DistanceFromHome',
 'Education',
 'EmployeeCount',
 'EmployeeNumber',
 'EnvironmentSatisfaction',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobSatisfaction',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StandardHours',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [15]:
categorical_cols

['Attrition',
 'BusinessTravel',
 'Department',
 'EducationField',
 'Gender',
 'JobRole',
 'MaritalStatus',
 'Over18',
 'OverTime']

In [17]:
# Variáveis categóricas ordinais codificadas numericamente
ordinal_cols = [
    'Education',
    'EnvironmentSatisfaction',
    'JobInvolvement',
    'JobSatisfaction',
    'PerformanceRating',
    'RelationshipSatisfaction',
    'WorkLifeBalance'
]

In [22]:
# Exclui features ordinais da lista numérica por representarem categorias ordenadas, não valores contínuos
numerical_cols = [
    col for col in numerical_cols
    if col not in ordinal_cols
]

In [24]:
numerical_cols

['Age',
 'DailyRate',
 'DistanceFromHome',
 'EmployeeCount',
 'EmployeeNumber',
 'HourlyRate',
 'JobLevel',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'PercentSalaryHike',
 'StandardHours',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [26]:
# Analisa a distribuição da variável alvo para compreender a taxa de attrition e avaliar o balanceamento entre as classes do problema
df['Attrition'].value_counts()

,count
Attrition,
No,1233
Yes,237


In [38]:
# Analisa a distribuição da variável alvo para compreender a taxa de attrition e avaliar o balanceamento entre as classes do problema
target_table = pd.DataFrame({
    'Contagem': df['Attrition'].value_counts(),
    'Taxa (%)': (df['Attrition'].value_counts(normalize=True) * 100).round(2)
})
target_table

,Contagem,Taxa (%)
Attrition,,
No,1233,83.88
Yes,237,16.12


In [39]:
# Annotation: A variável alvo apresenta desbalanceamento moderado, com predominância da classe negativa, o que deverá ser considerado nas etapas de modelagem e avaliação de performance

In [43]:
# Gera estatísticas descritivas das variáveis numéricas para avaliar distribuição, tendência central, dispersão e possíveis valores extremos
numerical_summary = df[numerical_cols].describe().T
numerical_summary

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.00,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
EmployeeCount,1470.0,1.000000,0.000000,1.0,1.00,1.0,1.00,1.0
EmployeeNumber,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.00,66.0,83.75,100.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.00,2.0,3.00,5.0
MonthlyIncome,1470.0,6502.931293,4707.956783,1009.0,2911.00,4919.0,8379.00,19999.0
MonthlyRate,1470.0,14313.103401,7117.786044,2094.0,8047.00,14235.5,20461.50,26999.0
NumCompaniesWorked,1470.0,2.693197,2.498009,0.0,1.00,2.0,4.00,9.0
